# 🧬 Co-evolutionary Fitness Landscape Explorer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elkins/synth-pdb/blob/master/examples/ml_integration/fitness_landscape_explorer.ipynb)

**Duration:** ~35 minutes | **Level:** ⭐⭐⭐ Advanced

---

## What is a fitness landscape?

Every protein sequence sits in a high-dimensional **fitness landscape** where
substituting one amino acid moves one step along one axis.
In this tutorial we map the **hydrophobic burial fitness landscape**:

1. Scan all 300 single-point mutants (15 positions × 20 AAs) using `burial_ratio`
2. Render a 2D heatmap where each cell is a (position, amino-acid) pair
3. Validate the landscape against the **Kyte-Doolittle hydrophobicity scale** —
   positions where burial is most sensitive to substitution should also show
   the strongest correlation between amino-acid hydrophobicity and burial score
4. Use `generate_msa_sequences()` to see how simulated neutral evolution samples
   the landscape under structural constraints


In [ ]:
# Install required dependencies for Colab / local Jupyter
!pip install -q py3Dmol biotite synth-pdb ipywidgets matplotlib

## 🔧 Setup


In [ ]:
import io
import os
import sys
import time

try:
    import google.colab

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess

    subprocess.run(["pip", "install", "-q", "synth-pdb", "biotite"], check=True)
else:
    repo_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", ".."))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)

import biotite.structure.io.pdb as pdb_io
import matplotlib.pyplot as plt
import numpy as np

from synth_pdb.evolution import generate_msa_sequences
from synth_pdb.generator import generate_pdb_content
from synth_pdb.validator import PDBValidator

AAS = list("ACDEFGHIKLMNPQRSTVWY")

# Kyte-Doolittle hydrophobicity scale (Kyte & Doolittle, 1982)
# Positive = hydrophobic, negative = hydrophilic
KD = {
    "A": 1.8,
    "R": -4.5,
    "N": -3.5,
    "D": -3.5,
    "C": 2.5,
    "Q": -3.5,
    "E": -3.5,
    "G": -0.4,
    "H": -3.2,
    "I": 4.5,
    "K": -3.9,
    "L": 3.8,
    "M": 1.9,
    "F": 2.8,
    "P": -1.6,
    "S": -0.8,
    "T": -0.7,
    "W": -0.9,
    "Y": -1.3,
    "V": 4.2,
}
KD_VEC = np.array([KD[aa] for aa in AAS])

print("Setup complete.")
print(f"Most hydrophobic: {max(KD, key=KD.get)} ({max(KD.values())})")
print(f"Most hydrophilic: {min(KD, key=KD.get)} ({min(KD.values())})")

## 🧬 Step 1: Wild-Type Baseline

`AAKLLLAAKLLLAAK` is a designed **amphipathic helix**:
- Positions 4,5,6,10,11,12 carry Leu — the hydrophobic face
- Positions 3,9,15 carry Lys — the polar face; terminal Ala are ambivalent

The WT `burial_ratio = 0.917` confirms good Oil Drop burial.


In [ ]:
WT_SEQ = "AAKLLLAAKLLLAAK"
N_POS = len(WT_SEQ)

print("Generating minimized wild-type...")
pdb_wt = generate_pdb_content(
    sequence_str=WT_SEQ, structure=f"1-{N_POS}:alpha", minimize_energy=True, seed=42
)

r_wt = PDBValidator(pdb_wt).get_quality_report()
print(f"burial_ratio : {r_wt['hydrophobic_burial_ratio']:.3f}")
print(f"Defensible   : {r_wt['is_overall_scientifically_defensible']}")

## 🔬 Step 2: Scan All 300 Mutants

For every (position, amino acid) pair we build the mutant alpha-helix and measure
`burial_ratio`. The fast path (no OpenMM) runs at ~190 structures/second.


In [ ]:
print(f"Scanning {N_POS} × {len(AAS)} = {N_POS * len(AAS)} mutants...")
t0 = time.time()

burial_matrix = np.zeros((len(AAS), N_POS), dtype=np.float32)

for pos in range(N_POS):
    for ai, aa in enumerate(AAS):
        mutant = WT_SEQ[:pos] + aa + WT_SEQ[pos + 1 :]
        pdb = generate_pdb_content(
            sequence_str=mutant, structure=f"1-{N_POS}:alpha", minimize_energy=False, seed=42
        )
        burial_matrix[ai, pos] = PDBValidator(pdb).calculate_residue_sasa()["burial_ratio"]

elapsed = time.time() - t0
print(f"Done: {N_POS * len(AAS)} mutants in {elapsed:.1f}s  ({N_POS * len(AAS) / elapsed:.0f}/s)")
print(f"burial_ratio range: [{burial_matrix.min():.3f}, {burial_matrix.max():.3f}]")
n_fail = (burial_matrix < 0.8).sum()
print(f"{n_fail}/{N_POS * len(AAS)} mutants fail Oil Drop threshold (< 0.8)")

## 🎨 Step 3: The Burial Fitness Heatmap

Green = good burial, Red = poor burial.
White dot = wild-type. 'x' = fails the 0.8 threshold.


In [ ]:
fig, ax = plt.subplots(figsize=(13, 6.5))

im = ax.imshow(
    burial_matrix, cmap="RdYlGn", vmin=0.65, vmax=1.15, aspect="auto", interpolation="nearest"
)

for pos, aa in enumerate(WT_SEQ):
    if aa in AAS:
        ax.plot(
            pos,
            AAS.index(aa),
            "o",
            color="white",
            markersize=7,
            markeredgecolor="#2c3e50",
            markeredgewidth=1.0,
            zorder=3,
        )

for ai in range(len(AAS)):
    for pos in range(N_POS):
        if burial_matrix[ai, pos] < 0.8:
            ax.text(
                pos,
                ai,
                "x",
                ha="center",
                va="center",
                fontsize=7,
                color="#922b21",
                fontweight="bold",
            )

ax.set_xticks(range(N_POS))
ax.set_xticklabels([f"{aa}\n{i + 1}" for i, aa in enumerate(WT_SEQ)], fontsize=8)
ax.set_yticks(range(len(AAS)))
ax.set_yticklabels(AAS, fontsize=8)
ax.set_xlabel("Position  (WT residue / number)", fontsize=10)
ax.set_ylabel("Substituted amino acid", fontsize=10)
ax.set_title(
    "Hydrophobic Burial Fitness Landscape\n"
    "Green = good burial  |  Red = poor burial  |  o = WT  |  x = fails threshold",
    fontsize=12,
    fontweight="bold",
)
plt.colorbar(im, ax=ax, label="burial_ratio", shrink=0.8)
plt.tight_layout()
plt.show()

## 📊 Step 4: Burial Sensitivity per Position

Positions with high standard deviation across the 20 AAs are **burial-sensitive** —
the fold's Oil Drop score depends strongly on what's placed there.


In [ ]:
pos_std = burial_matrix.std(axis=0)
pos_min = burial_matrix.min(axis=0)

fig, ax = plt.subplots(figsize=(12, 3.5))
colors = ["#e74c3c" if s > pos_std.mean() else "#3498db" for s in pos_std]
ax.bar(range(N_POS), pos_std, color=colors, alpha=0.85)
ax.axhline(
    pos_std.mean(),
    color="#7f8c8d",
    linestyle="--",
    linewidth=0.9,
    label=f"Mean ({pos_std.mean():.3f})",
)
ax.set_xticks(range(N_POS))
ax.set_xticklabels([f"{aa}{i + 1}" for i, aa in enumerate(WT_SEQ)], fontsize=8.5)
ax.set_ylabel("Burial sensitivity (std)", fontsize=9)
ax.set_title(
    "Per-Position Burial Sensitivity  (red = above average)", fontsize=11, fontweight="bold"
)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

top5 = np.argsort(pos_std)[::-1][:5]
print("Most burial-sensitive positions:")
for i in top5:
    print(
        f"  {WT_SEQ[i]}{i + 1}: std={pos_std[i]:.3f}, "
        f"range [{burial_matrix[:, i].min():.3f}, {burial_matrix[:, i].max():.3f}]"
    )

## 🔗 Step 5: Cross-Validation — Burial vs Hydrophobicity

Each column of the burial matrix is a profile across 20 AAs at one position.
We correlate this profile with the **Kyte-Doolittle hydrophobicity scale** (1982).

**Prediction:**
- At **interior positions**, placing a hydrophobic AA should *reduce* burial_ratio
  (all hydrophobic → polar surface shrinks → ratio drops). **Negative r.**
- At **terminal positions**, placing a bulky hydrophobic AA adds exposed surface,
  *raising* burial_ratio. **Positive r.**

This sign flip reveals the structural context of each position:


In [ ]:
# Pearson r between KD hydrophobicity and burial_ratio column at each position
r_vals = np.array([np.corrcoef(KD_VEC, burial_matrix[:, pos])[0, 1] for pos in range(N_POS)])

fig, ax = plt.subplots(figsize=(12, 4))
colors = ["#27ae60" if r > 0 else "#e74c3c" for r in r_vals]
bars = ax.bar(range(N_POS), r_vals, color=colors, alpha=0.85, edgecolor="white")
ax.axhline(0, color="#2c3e50", linewidth=1.0)
ax.axhline(0.5, color="#27ae60", linestyle=":", linewidth=0.8, alpha=0.7)
ax.axhline(-0.5, color="#e74c3c", linestyle=":", linewidth=0.8, alpha=0.7)
ax.set_xticks(range(N_POS))
ax.set_xticklabels([f"{aa}{i + 1}" for i, aa in enumerate(WT_SEQ)], fontsize=8.5)
ax.set_ylabel("Pearson r  (burial vs KD hydrophobicity)", fontsize=9)
ax.set_title(
    "Burial-Hydrophobicity Correlation per Position\n"
    "Green (r > 0): hydrophobic AAs improve burial  |  Red (r < 0): they reduce it",
    fontsize=11,
    fontweight="bold",
)
ax.set_ylim(-1, 1)
plt.tight_layout()
plt.show()

print("Per-position correlation (burial_ratio vs KD hydrophobicity):")
for i, (aa, r) in enumerate(zip(WT_SEQ, r_vals, strict=False)):
    sign = (
        "+hydrophobic improves burial"
        if r > 0.3
        else ("-hydrophobic hurts burial" if r < -0.3 else "weak signal")
    )
    print(f"  {aa}{i + 1:2d}: r={r:+.3f}  {sign}")

## 🌳 Step 6: Neutral Evolution via `generate_msa_sequences()`

Finally, we use **simulated neutral evolution** to generate a synthetic MSA.
The module applies a structural constraint: buried residues (low rSASA) can
only mutate to hydrophobic residues, while surface residues drift freely.

For this short helix all residues compute as surface-exposed, so the MSA
represents unconstrained neutral drift — a useful baseline showing what sequence
diversity looks like *without* strong structural selection pressure.


In [ ]:
pdb_file = pdb_io.PDBFile.read(io.StringIO(pdb_wt))
structure = pdb_file.get_structure(model=1)

print("Generating 500 synthetic homologs via neutral drift...")
msa = generate_msa_sequences(structure, n_seqs=500, mutation_rate=0.15, conservation_threshold=0.2)

# Per-position amino-acid frequency heatmap
msa_arr = np.array([list(s) for s in msa])
freq_matrix = np.zeros((len(AAS), N_POS))
for pos in range(N_POS):
    col = msa_arr[:, pos]
    for ai, aa in enumerate(AAS):
        freq_matrix[ai, pos] = np.sum(col == aa) / len(col)

fig, ax = plt.subplots(figsize=(13, 6))
im = ax.imshow(freq_matrix, cmap="Blues", vmin=0, vmax=0.25, aspect="auto", interpolation="nearest")
for pos, aa in enumerate(WT_SEQ):
    if aa in AAS:
        ax.plot(
            pos,
            AAS.index(aa),
            "*",
            color="#e74c3c",
            markersize=9,
            markeredgecolor="black",
            markeredgewidth=0.5,
            zorder=3,
        )
ax.set_xticks(range(N_POS))
ax.set_xticklabels([f"{aa}{i + 1}" for i, aa in enumerate(WT_SEQ)], fontsize=8)
ax.set_yticks(range(len(AAS)))
ax.set_yticklabels(AAS, fontsize=8)
ax.set_xlabel("Position", fontsize=10)
ax.set_ylabel("Amino acid", fontsize=10)
ax.set_title(
    "MSA Frequency Matrix (500 neutral-drift homologs)\n"
    "Darker = more frequent in simulated evolution  |  * = wild-type",
    fontsize=11,
    fontweight="bold",
)
plt.colorbar(im, ax=ax, label="Frequency", shrink=0.8)
plt.tight_layout()
plt.show()

# Sequence diversity statistics
from collections import Counter

print(f"MSA diversity: {len(set(msa))} unique sequences / 500")
print(f"Most common: {Counter(msa).most_common(1)[0][1]}/500 identical copies")
print("Example homologs:")
print(f"  WT  : {WT_SEQ}")
for s in msa[:3]:
    diffs = sum(1 for a, b in zip(WT_SEQ, s, strict=False) if a != b)
    print(f"  Seq : {s}  ({diffs} mutations)")

## 🎓 Key Takeaways

### The burial fitness landscape
The heatmap encodes a **structural substitution matrix** specific to this sequence
and fold:
- Columns that are uniformly green: the position tolerates any substitution
- Columns with red patches: specific amino acids collapse the Oil Drop burial
- Cells marked 'x' fail the 0.8 threshold outright

### The hydrophobicity cross-validation (Step 5)
The Pearson r values reveal **structural context**:
- **r < 0** (red bars): placing a hydrophobic AA at this position crowds out
  polar surface area — reducing burial_ratio. These are positions where
  *polar residues help* satisfy the Oil Drop.
- **r > 0** (green bars): placing a hydrophobic AA adds to the exposed
  surface, paradoxically raising burial_ratio — typical of terminal positions
  where the helix frays and all residues are exposed.

The **sign flip** is a real structural signal, not an artefact — it reflects
the different roles of the N/C-terminal vs central helix positions.

### `generate_msa_sequences()` as a generation tool
The MSA simulator generates structurally-constrained sequence diversity.
For a short exposed helix, the constraint is loose (all residues surface-exposed),
producing high diversity — useful for training ML models on sequence variation.
For a longer, more globular protein with a buried hydrophobic core, the buried
positions would be locked to CORE_ALLOWED = {A,V,I,L,M,F,W,Y}.

### What to try next
- Extend to **30 residues**: the periodic burial pattern of a helix (3.6 res/turn)
  becomes clearly visible as diagonal bands in the heatmap
- Combine the scan with the **Contact Map Fingerprinting** tutorial: score each
  mutant by how much its contact map diverges from the WT fingerprint
- Apply to a **real sequence** from the PDB to map experimental mutational
  tolerance data against the computational landscape
